# 01.4 The Taxonomy of Learning Problems

> **Prerequisites:** 01.1 (the late-payment model, precision@k, queue capacity) ·
> 01.2 (the data universe and its grain) · 01.3 (manifests, and recording config)
> **What you'll learn:**
> - Frame one business question four different ways and show the framings disagree in practice
> - Choose between a probability threshold and a capacity-bounded selection from the constraint
> - Read an estimator's API surface to infer what problem it solves, and where that inference breaks
> - Price the parametric/non-parametric and batch/online choices in artifact bytes and microseconds
> - Recognize when supervised learning is unavailable because the labels do not exist yet
> **Level:** Beginner · **Series:** 01 The ML Landscape & Project Lifecycle

> ⚡ **Monday 2026-04-13, 09:05** — the dunning queue lands with 1,773 invoices and three
> analysts to work them. The previous Tuesday the same job produced 241 and the team went home
> early. Nobody changed the model, the data or the threshold. The cause: the queue is defined by
> a cutoff on a probability, and the constraint it has to satisfy is a headcount.

## Concept
### Plain-English Explanation

Textbooks present the taxonomy of machine learning as a filing system: supervised over here,
unsupervised over there, regression predicts numbers and classification predicts categories.
Filed that way it is a vocabulary quiz, and it is why the topic gets skimmed.

The engineering version is that the taxonomy is a set of **commitments**. Choosing a framing
fixes three things simultaneously: what data you must supply before you can train at all, what
you are able to measure afterwards, and what the running system is contractually obliged to
return. Those commitments are what actually bite. The cold open is not a modelling failure — the
model is fine, and 01.1 measured it carefully — it is a framing failure, where a system built to
answer "how likely is this invoice to be late?" was installed where the business had asked
"which three hundred should we chase today?"

Those two questions have the same features, the same data and nearly the same model. They differ
in what the system returns, and that difference is what put three analysts in front of 1,773
invoices on a Monday morning.

### Technical Explanation

Four axes matter, and each is a decision with a measurable cost.

**What supervises the learning.** *Supervised* learning needs a target column that already
exists for historical rows. *Unsupervised* learning needs none. That sounds like a preference
until you look at when labels arrive: PayFlow's invoices are 95.7% labelled overall, but only
21.2% of the invoices issued in the most recent month have a resolved payment outcome. The rows
a live model must score are precisely the rows whose answers are not yet known, and that lag —
not philosophy — is the practical reason unsupervised and self-supervised methods exist.

**What shape the output takes.** The same PayFlow question supports at least four targets:
`days_late` as a continuous number (regression), `late = days_late > 7` as a label
(classification), the same label but with only the *order* mattering (ranking), and no target at
all (clustering). ⭐ **CRITICAL CONCEPT** — these are not cosmetic relabelings of one problem.
Ranking the same invoices by predicted `days_late` and by predicted `P(late)` produces queues
that disagree on 27.0% of their slots, so the framing choice changes which work actually gets
done even when the aggregate metrics are nearly identical.

**When learning happens.** *Batch* training fits on an accumulated dataset and ships an artifact;
*online* learning updates incrementally through `partial_fit` as data arrives. Online sounds
strictly better and is not: fitting quarter by quarter on this problem reaches a precision@k of
0.3681, against 0.4462 for the batch fit on the same features.

**What the model is.** A *parametric* model compresses its training data into a fixed set of
coefficients; a *non-parametric* model keeps the data and consults it at prediction time. That
distinction is abstract until it is weighed: the logistic regression here serializes to 3.8 KB,
while a fifteen-neighbour KNN fitted on the same 20,000 training rows serializes to 5,003.5 KB
and takes about ten times as long per prediction — over a thousand times the artifact, because
the non-parametric model *is* its training data. (Artifact sizes are deterministic; the absolute
microsecond figures below are machine- and run-dependent, so the ratio is the number to carry.)

### Mental Model

A framing is a contract with three clauses: what you must supply (labels, and when they arrive),
what you may measure (the metric that is even defined), and what the system must return (a
number, a probability, exactly *k* identifiers, or a group id). Choose it from the operational
constraint, not from the algorithm you happen to like.

## How It Works

```text
              ONE PayFlow invoice table (159,821 train rows x 10 feature columns)
                                       |
        +---------------+--------------+--------------+----------------+
        v               v                             v                v
   regression      classification              ranking@k          clustering
   y = days_late   y = late (0/1)              y = late, order    y = none
        |               |                      only                    |
   MAE / RMSE      PR-AUC, precision@k         precision@k        silhouette, stability
        |               |                      |                       |
   returns a       returns a probability       returns EXACTLY k   returns a group id
   number          in [0,1]                    identifiers        with no notion of
                                                                  "correct"

   threshold selection                        capacity selection
   ------------------                         ------------------
   fix t, queue = #{score >= t}               fix k, t floats to whatever admits k
   queue SIZE is an outcome                   queue SIZE is a guarantee
   t=0.3 -> 1,773   t=0.4 -> 818              k=300 -> exactly 300, every day
   t=0.5 ->   424   t=0.6 -> 241
```

The bottom block is the mechanism behind the cold open, and it is worth stating precisely. A
threshold fixes a point on the score axis; the queue size is then whatever probability mass
happens to lie above that point. That mass moves whenever the score distribution moves — with
customer mix, with seasonality, and, as 01.1 showed, with a payment-gateway migration that
shifts the underlying late rate. Capacity selection inverts the relationship: fix the number of
slots and let the implied threshold float to whatever value admits exactly that many.

The two are duals, and only one of them matches a headcount. Note from the numbers above that no
fixed threshold solves the problem either, because the queue swings from 818 to 241 across a
single step from 0.4 to 0.6 — there is no value of *t* that reliably yields 300, and searching
for one is the wrong activity.

The other three axes carry costs that are equally concrete: labels that arrive late constrain
what can be supervised at all, artifact size and prediction latency constrain what can be
served, and incremental updates trade accuracy for freshness. None of these are visible if the
taxonomy is treated as vocabulary.

## Hands-On Build
### Stage A — from scratch

Take one slice of PayFlow invoices and construct all four learning problems from it by hand. The
features never change. Only the target changes — and with it, what can be measured and what the
system returns.

In [1]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd

LAB = Path.cwd() / "_lab" / "lab_01.4_taxonomy.py"
spec = importlib.util.spec_from_file_location("lab_01_4", LAB)
lab = importlib.util.module_from_spec(spec)
sys.modules["lab_01_4"] = lab
spec.loader.exec_module(lab)

df, _ = lab.lab11.build_dataset()
w = lab.lab11.windows(df)
train, test = w["train"], w["test_stable"].head(5_500)     # ~one month of invoices
k = int(lab.lab11.dunning_rule(test).sum())

# Four targets from one table. Note KMeans is fitted WITHOUT a y argument at all.
preds = lab.build_framings(train, test)

  feature matrix: train (159821, 10), test (5500, 10)  (6 numeric + 4 categorical columns)

  framing                 target                        evaluated with        returns
  regression              days_late (int, unbounded)    MAE / RMSE            a number
  binary classification   late = days_late > 7 (0/1)    PR-AUC, precision@k   a probability
  ranking under capacity  late, but only order matters  precision@k           exactly k ids
  clustering              no target at all              silhouette, stability a group id



  clustering assigned 4 groups, sizes [669, 2817, 53, 1961] - no notion of 'correct' anywhere


The feature matrix is identical across all four — 159,821 training rows and ten columns, six
numeric and four categorical. What differs is the fourth column of that table: regression returns
a number, classification a probability, ranking exactly *k* identifiers, and clustering a group
id whose correctness is undefined. The clustering run partitions the evaluation window into
groups of 669, 2,817, 53 and 1,961 invoices, and there is no answer key anywhere against which
those groups could be scored — series 21 has to build an entirely different notion of evaluation
for exactly this reason.

The interesting question is whether the two supervised framings, which share features and
training rows and differ only in what they call the target, actually behave differently.

In [2]:
lab.framing_disagreement(test, preds, k)

  top-1,524 by predicted days_late vs by predicted P(late):
    agree on 1,113 of 1,524 slots (73.0%); 411 slots (27.0%) differ
    regression top-k       precision=0.4423  value=$  11,474
    classification top-k   precision=0.4462  value=$  11,350
    precision gap between the two framings: 0.0039  -> inside the 01.3 noise band, while 27% of the WORK differs


They disagree on 411 of 1,524 queue slots — 27.0% of the work — while scoring almost identically
in aggregate: precision 0.4423 for the regression framing against 0.4462 for classification, and
value captured of $11,474 against $11,350. ⚠️ Read those two facts together, because they are
easy to mistake for one another. The framings are nearly equivalent *as measured*, and
substantially different *in what they do*. More than a quarter of the analysts' day is spent on
different invoices depending on a choice that a summary metric reports as a rounding difference.

That is also why the regression framing captures slightly more value despite slightly lower
precision: ranking by predicted lateness in days is a different ordering from ranking by
probability of crossing a threshold, and it happens to surface invoices that are later, which is
where the money is — the same objective-versus-proxy tension 01.1 measured.

### Stage B — idiomatic

The taxonomy is not just a mental model; scikit-learn encodes it in the estimator API. Whether
`fit` requires a `y`, and whether `predict_proba`, `transform` or `partial_fit` exist, is a
readable signature of what kind of problem an estimator solves.

In [3]:
lab.api_surface()

  estimator               needs y   predict   proba    transform   partial_fit  implied problem
  LinearRegression        True      True      False    False       False        regression
  LogisticRegression      True      True      True     False       False        classification
  SGDClassifier(hinge)    True      True      False    False       True         regression (online-capable)
  SGDClassifier(log_loss) True      True      True     False       True         classification (online-capable)
  KNeighborsClassifier    True      True      True     False       False        classification
  KMeans                  False     True      False    True        False        unsupervised
  PCA                     False     False     False    True        False        unsupervised

  Note the two SGDClassifier rows: identical class, different `loss`, and
  predict_proba EXISTS on one and not the other (scikit-learn gates it with
  available_if). The naive rule above therefore mislabels the hing

The pattern is mostly clean: estimators whose `fit` has no required `y` are unsupervised and
expose `transform`; those exposing `predict_proba` are classifiers; those exposing `partial_fit`
can learn online.

⚠️ Then read the two `SGDClassifier` rows. Same class, same task, different `loss` — and
`predict_proba` exists on the `log_loss` variant and genuinely does not exist on the `hinge`
one, because scikit-learn gates the method with `available_if`. The naive rule therefore
classifies the hinge variant as a regressor, which is wrong. This matters beyond trivia: code
that does `if hasattr(model, "predict_proba")` to decide how to handle an estimator will silently
take the wrong branch depending on a hyperparameter set somewhere else entirely. The API surface
is strong evidence about the problem type, not proof of it.

### Stage C — production

Each framing implies a different serving contract, and the contract is where the cold open was
lost. A classifier's contract is "return a probability"; the collections system needed "return
exactly 300 identifiers." Below, the constraint is priced — first the framing that must be
served, then what serving it costs.

In [4]:
lab.threshold_vs_capacity(test, preds["classification"])

  evaluation window: 5,500 invoices, actual late rate 0.268
  model mean predicted P(late): 0.272, max 0.799
  invoices scoring >= 0.5: 424  -> a queue of 424 against a capacity of 300/day
    threshold 0.3 -> queue of  1,773  (floods a 300-slot team)
    threshold 0.4 -> queue of    818  (floods a 300-slot team)
    threshold 0.5 -> queue of    424  (floods a 300-slot team)
    threshold 0.6 -> queue of    241  (starves a 300-slot team)
  top-300 selection instead: queue=300 exactly, precision=0.6333 - the size is a guarantee, not an outcome


No threshold delivers the contract. At 0.3 the queue is 1,773 invoices; at 0.6 it is 241; the
crossing happens inside one step of a coarse sweep, and every one of those numbers moves again
whenever the score distribution shifts. Selecting the top 300 instead returns 300 every single
day by construction, and — because the very highest-scoring invoices are the most likely to be
late — reaches a precision of 0.6333 on this window, well above anything the threshold sweep
produced.

The remaining production choices are the model's shape and its update cadence, both of which
have prices that are easy to measure and easy to forget until a latency budget is missed.

In [5]:
lab.production_costs(train, test)

  LogisticRegression (parametric)    artifact       3.8 KB   predict     3.5 us/row


  KNeighbors k=15 (non-parametric)   artifact   5,003.5 KB   predict    39.5 us/row
  ratios (non-parametric / parametric): artifact 1,326x   latency 11.1x
  (fitted on 20,000 rows, timed over 2,000 predictions;
   absolute microseconds are machine- and run-dependent, the ratio is not)
  the non-parametric model IS its training data: the rows ship to production



  online learning: partial_fit over successive quarters, no full refit
    after 2023Q3: n=11,121  precision@k=0.2507
    after 2023Q4: n=11,799  precision@k=0.3865
    after 2024Q1: n=12,750  precision@k=0.3248
    after 2024Q2: n=13,721  precision@k=0.3871


    after 2024Q3: n=14,648  precision@k=0.3773
    after 2024Q4: n=15,586  precision@k=0.3681


The parametric model serializes to 3.8 KB; the non-parametric one, fitted on the same 20,000
rows, serializes to 5,003.5 KB and costs roughly ten times as much per prediction — because a
KNN artifact literally contains the training rows and consults them at prediction time. Every
retrain ships the dataset to production again, and the artifact grows with the business rather
than staying constant. Note that the notebook re-runs this timing on whatever machine executes
it, so the microsecond figures will differ from run to run while the ratio holds; that is 01.3's
determinism lesson applied to a benchmark rather than to a model.

Online learning is the other trade. Feeding quarters sequentially through `partial_fit` avoids
full refits and keeps the model close to recent data, and its precision@k wanders between 0.2507
and 0.3871 without ever reaching the batch fit's 0.4462. The freshness is real and so is the
cost; neither is free, and both are measurable before committing.

## Evaluation

For a framing notebook the thing being evaluated is the framing itself, so the harness is the
comparison already run: identical features and rows, four targets, measured on precision@k and
on the value metric from 01.1, plus the queue-size contract each framing can honour. The baseline
is the classification-with-threshold design that produced the incident.

The result is that framings which look interchangeable on a summary metric are not
interchangeable in operation. Regression and classification differ by 0.0039 in precision — well
inside the noise band 01.3 established for this pipeline, so on that evidence alone they are
indistinguishable — while disagreeing on 27.0% of the actual work. And the framing that matches
the constraint, capacity selection, reaches 0.6333 precision at exactly the required queue size,
which is not a modelling improvement at all: the model is unchanged, and only the selection rule
was corrected.

A meaningful delta here is therefore not a metric movement. It is whether the serving contract
can be honoured every day, and the threshold design cannot honour it at any parameter value.

## Design Patterns / Tradeoffs

**Threshold selection versus capacity selection.** A threshold answers "is this invoice likely
enough to act on?", which is the right question when each decision is independent and there is no
shared constraint — an automated email that costs the same whether it is sent to 200 or 2,000
recipients, a fraud block that must fire on its own merits. Its failure mode is exactly the cold
open: when a fixed resource is consumed, queue size becomes an uncontrolled output that drifts
with the score distribution. Capacity selection answers "which *k* should we act on?", guarantees
the queue size, and needs no calibration at all because only the ordering matters. Its weakness
is that it always returns *k* items even on a day when only three invoices are genuinely at risk,
so it must be paired with a floor threshold when acting on a low-risk item has real cost. Use
capacity selection whenever a human queue, a budget or a rate limit is in the loop; use
thresholds when actions are independent and cheap; use both together when neither the volume nor
the quality can be allowed to float.

**Parametric versus non-parametric.** Parametric models compress the training data into fixed
coefficients: small artifacts, constant prediction latency, and an artifact whose size does not
grow with the business — 3.8 KB here. They pay for that with a fixed functional form that can
underfit genuinely complex boundaries. Non-parametric models keep the data and stay flexible, at
a cost that is measurable and often disqualifying: 5,003.5 KB and an order of magnitude more
latency per row on a mere 20,000 training rows, both scaling with the dataset, plus the operational
awkwardness of shipping training data into the serving tier — which is also a privacy surface,
since the artifact contains customer rows. Use non-parametric methods when the boundary is
irregular and the dataset is small and stable; avoid them when latency budgets are tight, when
the data grows continuously, or when the artifact crosses a trust boundary.

**Batch versus online.** Batch training is reproducible in the sense 01.3 demanded — a manifest
plus a data snapshot regenerates the artifact exactly — and it is easier to evaluate, because the
model is a fixed object. Online learning adapts continuously, which is the honest answer to the
drift 01.1 diagnosed, but it makes "which model produced this prediction?" a question about a
moment in time rather than about an artifact, and here it costs real accuracy: 0.3681 after six
quarters against 0.4462 for batch. Prefer scheduled batch retraining with drift monitoring as the
default; reach for online updates when the environment moves faster than a retraining cycle and
you can afford the evaluation machinery to keep watching a moving object.

**Recommendation for PayFlow:** frame the dunning problem as ranking under capacity, train in
batch on a snapshot, serve a parametric model, and record the framing in the manifest's config
block so the serving contract is part of the run's identity rather than a convention in
somebody's head.

## Production Scenario
### Symptoms

**Monday 2026-04-13, 09:05.** The dunning model has been live since March, selecting the queue by
thresholding predicted probability at 0.5.

- **09:05** — the daily queue contains 1,773 invoices. The collections team is three people.
- The previous Tuesday's queue held 241 items and the team ran out of work before lunch. The
  swing has no explanation anybody can offer, and there is no ticket, because on the light day it
  looked like good news.
- The model dashboard is **green**: precision on the worked subset is healthy and unremarkable.
  There is no model-quality alert to fire, because model quality is not what broke.
- Deployment history shows no release. The threshold has not been touched since March. Feature
  distributions and null rates are unchanged.
- Collections' own SLA metric — invoices contacted within two days of falling overdue — has been
  degrading for weeks, because on flood days the team cannot get through the list and the excess
  is silently dropped rather than deferred.

In [6]:
# The score distribution the queue is carved out of, and what a fixed cutoff does to it.
lab.threshold_vs_capacity(test, preds["classification"])
print()
lab.label_availability()

  evaluation window: 5,500 invoices, actual late rate 0.268
  model mean predicted P(late): 0.272, max 0.799
  invoices scoring >= 0.5: 424  -> a queue of 424 against a capacity of 300/day
    threshold 0.3 -> queue of  1,773  (floods a 300-slot team)
    threshold 0.4 -> queue of    818  (floods a 300-slot team)
    threshold 0.5 -> queue of    424  (floods a 300-slot team)
    threshold 0.6 -> queue of    241  (starves a 300-slot team)
  top-300 selection instead: queue=300 exactly, precision=0.6333 - the size is a guarantee, not an outcome



  share of invoices with a resolved payment outcome, by issue month:
    2025-12  n= 6,175  labelled=98.2%  #######################################
    2026-01  n= 6,307  labelled=98.1%  #######################################
    2026-02  n= 6,483  labelled=98.1%  #######################################
    2026-03  n= 6,718  labelled=98.0%  #######################################
    2026-04  n= 6,881  labelled=97.8%  #######################################
    2026-05  n= 7,045  labelled=98.2%  #######################################
    2026-06  n= 7,207  labelled=96.2%  ######################################
    2026-07  n= 7,301  labelled=83.8%  #################################
    2026-08  n= 7,234  labelled=21.2%  ########

  overall labelled: 95.7% of 288,040 invoices
  the newest rows - the ones a live model must score - are the least labelled


### Diagnosis

1. **Alert** — a staffing and SLA complaint rather than an ML alert: the queue is unworkable on
   some days and empty on others. Candidate causes: a data volume change, a model change, a
   threshold change, or a framing mismatch.
2. **Queue-size dashboard** — the queue is not stable and never was; its size ranges over an
   order of magnitude across weeks. This eliminates "something broke last Friday" and reframes
   the question as why queue size was ever expected to be stable.
3. **Prediction logs** — the score distribution is doing what score distributions do: drifting
   with customer mix and seasonality. Mean predicted lateness on this window is 0.272 against an
   actual rate of 0.268, so the model is well calibrated and not at fault. The cutoff at 0.5
   admits 424 invoices here, 1,773 at 0.3, 241 at 0.6 — the queue is an integral over a moving
   density, which is not a controlled quantity.
4. **Input-data checks** — schema, null rates and volumes unchanged, eliminating an upstream data
   fault. Label availability shows the expected censoring gradient, 98.2% for December against
   21.2% for the most recent month, confirming the evaluation window was chosen correctly and is
   not itself the problem.
5. **Version diff** — no model release, no threshold change, no feature change. Nothing broke.
   The system is behaving exactly as designed, which is the finding.
6. **Design review** — the requirement stated to the team was "flag invoices likely to be paid
   late"; the requirement the business had was "give three analysts a full day of the highest-risk
   work." The first is a classification contract, the second is a ranking-under-capacity contract,
   and the gap between them was never written down anywhere.

### Root Cause

The system was framed as binary classification with a fixed decision threshold, so queue size is
an output — the probability mass above 0.5 — rather than an input. The business constraint is a
headcount, which fixes queue size and leaves the threshold free. Because the two formulations are
duals, no value of the threshold can satisfy a capacity constraint: the queue swings from 818 at
0.4 to 241 at 0.6, and it moves again whenever the score distribution shifts.

### Fix

**Mitigation now.** Change the selection rule from `scores >= 0.5` to top-*k* with *k* set to the
team's capacity. It is a one-line change to the scoring job, requires no retraining, guarantees
exactly 300 items every day, and on this window raises precision from the threshold design to
0.6333 because the highest-scoring invoices are genuinely the riskiest.

**Permanent fix.** Reframe the problem and make the framing explicit everywhere it is load-bearing.
The serving contract becomes "return exactly *k* invoice ids, ordered", the evaluation metric
becomes precision@k at the production *k* rather than a threshold-based score, and the framing plus
*k* are recorded in the manifest's config block from 01.3 so that a future reader can tell what
contract a given artifact was built to satisfy. Where a low-risk floor matters, pair the top-*k*
selection with a minimum score so that a genuinely quiet day does not manufacture work.

### Prevention

- **An acceptance test that asserts the contract**: `len(queue) == capacity` on every scheduled
  run. Trivial to write, and it fails on the first flood day rather than after weeks of SLA decay.
- **Alert on queue-size variance, not only on model quality.** The model dashboard was green
  throughout because the model was fine; the missing signal was about the system's output shape.
- **Record the framing in the run manifest.** A config block naming `ranking@k` with its *k* makes
  the serving contract part of the artifact's identity, so a mismatch is visible in a diff rather
  than in a staffing complaint.
- **Write the constraint into the requirement.** "Flag likely-late invoices" and "fill three
  analysts' day with the riskiest work" are different specifications; the second names a capacity
  and would have selected the right framing on day one.

## Common Pitfalls

⚠️ **Thresholding at 0.5 because it is the default.** Nobody chooses 0.5; it arrives with
`predict`. It is only meaningful when the classes are balanced and the costs are symmetric, which
is almost never true — and when a capacity constraint exists it is not a defensible choice at any
value.

**Treating a regression output as a probability.** A model trained to predict `days_late` emits
an unbounded number; comparing it to 0.5, or averaging it as if it were a likelihood, produces
confident nonsense. Check what the target was before interpreting the output.

⚠️ **Testing `hasattr(model, "predict_proba")` to infer the problem type.** The two
`SGDClassifier` rows differ only in `loss`, and the method genuinely exists on one and not the
other. Dispatching on method presence makes behaviour depend on a hyperparameter chosen
elsewhere; dispatch on an explicit, recorded framing instead.

**Choosing the algorithm before the framing.** "We should use gradient boosting" is a statement
about a solution to an unstated problem. The framing determines what labels are needed, what
metric is defined and what the system returns — all of which constrain the algorithm, not the
other way round.

**Evaluating a ranker with a classification metric.** Accuracy and threshold-based scores answer
a question the ranker was never asked. If the system returns *k* items, the metric is computed at
*k*.

**Assuming labels exist because a column exists.** Only 21.2% of the most recent month's invoices
have a resolved outcome. A supervised framing needs its labels to have arrived, and the newest
rows — the ones being scored — are the least labelled, which bounds how quickly any supervised
system can learn about a change.

**Adopting non-parametric methods without pricing them.** The KNN artifact here is over a
thousand times larger than the parametric one and nine times slower per prediction, and it
carries customer rows into the serving tier. Measure artifact size and latency before the
architecture is fixed.

## Interview Questions

1. **Derive this.** Show that a fixed threshold cannot guarantee a queue of fixed size, and give
   the relationship between the threshold and the queue size that a capacity-based selector
   inverts. *Answer shape:* queue size is the count of scores above *t*, which is the survival
   function of the score distribution evaluated at *t*, times *n*; it therefore moves whenever
   that distribution or *n* moves. Capacity selection fixes the count and takes the implied
   threshold as the *k*-th order statistic, which floats freely.
2. **Design this.** Collections has three analysts and wants the riskiest invoices worked daily,
   but does not want anyone contacted whose risk is trivial. Specify the framing, the serving
   contract, the metric and the tests. *Answer shape:* ranking under capacity with a floor
   threshold; contract returns at most *k* ordered ids and may return fewer if the floor is not
   met; metric is precision@k at production *k*, reported beside a baseline; acceptance test
   asserts the size contract; framing and *k* recorded in the manifest.
3. **Debug this.** A daily queue's size swings by an order of magnitude week to week with no
   deploys, no data faults and a healthy model dashboard. Diagnose. *Answer shape:* check whether
   size was ever controlled — if selection is threshold-based it is an output, not an input; verify
   calibration to rule out model fault; confirm no version change; then treat it as a framing
   mismatch between a probability contract and a capacity constraint.
4. The same features and rows, framed as regression and as classification, give precision within
   0.0039 of each other but disagree on 27.0% of the selected items. What do you conclude?
   *Answer shape:* aggregate metrics measure quality, not agreement; the framings are
   statistically indistinguishable here yet operationally different, so the choice should be made
   on the serving contract and on which ordering better matches the objective, not on the metric.
5. When is unsupervised learning the right answer rather than a fallback? *Answer shape:* when no
   label exists or labels arrive far too late to act on — 21.2% labelled in the most recent month
   here; when the task genuinely is structure discovery, such as segmentation with no ground
   truth; or when labelling cost dominates. Note that it buys a harder evaluation problem, since
   there is nothing to be correct against.
6. What does `partial_fit` buy and what does it cost? *Answer shape:* freshness without full
   refits, at the cost of accuracy on this problem (0.3681 against 0.4462), reproducibility — the
   model is a moving object rather than an artifact — and evaluation complexity, since every
   measurement is of a different model.
7. Your KNN prototype beats logistic regression on offline metrics. What do you check before
   proposing it for production? *Answer shape:* artifact size and its growth with data, per-row
   prediction latency against the budget, memory in the serving tier, retraining and deployment
   cost, and whether shipping training rows into production crosses a privacy or trust boundary.

## Key Takeaways

- A framing is a contract fixing what you must supply, what you may measure and what the system
  must return; choose it from the operational constraint, not from a preferred algorithm.
- Threshold selection makes queue size an output and capacity selection makes it a guarantee —
  no threshold value could deliver 300 slots, with the queue swinging from 1,773 to 241.
- Framings that look equivalent on a summary metric need not be operationally equivalent: 0.0039
  apart on precision, 27.0% apart on which invoices get worked.
- Fixing the framing rather than the model raised precision to 0.6333 at exactly the required
  queue size, with no retraining at all.
- The estimator API is strong evidence of the problem type but not proof — `predict_proba` exists
  on `SGDClassifier(log_loss)` and not on `SGDClassifier(hinge)`.
- Supervised learning requires labels that have arrived: 95.7% of invoices are labelled overall
  but only 21.2% of the most recent month, and the newest rows are the ones being scored.
- Non-parametric models are their training data — 5,003.5 KB against 3.8 KB, and roughly ten
  times the per-row latency — so price them before the architecture is fixed.
- Online learning buys freshness and costs accuracy and reproducibility; batch with drift
  monitoring is the better default until the environment outruns the retraining cycle.

## Related

**Backward**

- **01.1 Rules or Learning?** — established the queue and its capacity *k*; the framing mismatch
  diagnosed here is why that notebook compared policies at matched *k* rather than at a threshold.
- **01.2 First Contact with the PayFlow Data Universe** — the grain of the invoice table is what
  makes one row one prediction, and therefore what makes these four framings available at all.
- **01.3 Reproducibility as an Engineering Contract** — supplies the manifest whose config block
  is where the framing and *k* belong, and the noise band that makes the 0.0039 precision gap
  readable as indistinguishable.

**Forward**

- **01.5 Problem Framing** — takes the next step: given that a framing is a choice, how the label
  itself is designed (horizon, population, cutoff) and how much the choice moves the base rate.
- **01.6 The ML Project Lifecycle** — where framing sits relative to data collection, evaluation
  and deployment in the sequence of a project.
- **11.1 Linear Regression** — the regression framing used here as a black box, derived properly.
- **15.1 Logistic Regression & Classifier Practice** — canonical home for thresholds, calibration,
  class imbalance and the cost matrices that decide where a floor threshold belongs.
- **16.1 K-Nearest Neighbors** — the non-parametric model priced here, with the distance metrics
  and indexing structures that make its latency tractable.
- **21.1 Clustering** — how to evaluate the framing that has no ground truth, including the
  stability and silhouette measures named but not used here.
- **34.1 ML in Production** — serving contracts, acceptance tests on output shape, and the
  monitoring that would have caught a queue-size swing.